In [1]:
# Verify that this notebook is running inside
# the clean Project 1 reproducibility environment.

import sys
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib
import matplotlib
import openpyxl


print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nOperating system:")
print(platform.platform())

print("\nPackage versions:")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("matplotlib:", matplotlib.__version__)
print("openpyxl:", openpyxl.__version__)


# Confirm that Python is running from the clean environment.
assert ".venv_repro" in sys.executable.lower(), (
    "The notebook is not using the clean reproducibility environment."
)

print(
    "\nClean reproducibility kernel verified successfully."
)

Python executable:
C:\GEO_Breast_Reliability\.venv_repro\Scripts\python.exe

Python version:
3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]

Operating system:
Windows-11-10.0.26200-SP0

Package versions:
numpy: 2.4.5
pandas: 3.0.3
scipy: 1.17.1
scikit-learn: 1.8.0
joblib: 1.5.3
matplotlib: 3.10.9
openpyxl: 3.1.5

Clean reproducibility kernel verified successfully.


In [3]:
# Clean-environment verification of the saved
# 405-tumor GSE81538 model and external-validation results.

from pathlib import Path

import hashlib
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    roc_auc_score,
)


# =========================================================
# 1. Define project paths
# =========================================================

BASE_DIR = Path(
    r"C:\GEO_Breast_Reliability"
)

PROCESSED_DIR = (
    BASE_DIR
    / "Processed"
)

EXTERNAL_RESULTS_DIR = (
    BASE_DIR
    / "Results"
    / "external_sensitivity_405"
)

MODELS_DIR = (
    EXTERNAL_RESULTS_DIR
    / "models"
)

REPRODUCIBILITY_DIR = (
    BASE_DIR
    / "reproducibility"
)


MODEL_BUNDLE_PATH = (
    MODELS_DIR
    / "GSE81538_405_ER_prediction_model_bundle.joblib"
)

METADATA_PATH = (
    MODELS_DIR
    / "GSE81538_405_ER_prediction_model_metadata.json"
)

SHARED_GENES_PATH = (
    MODELS_DIR
    / "GSE81538_GSE96058_shared_genes_405.csv"
)

CHECKSUM_PATH = (
    REPRODUCIBILITY_DIR
    / "model_artifact_sha256_checksums.csv"
)

SAVED_PREDICTIONS_PATH = (
    EXTERNAL_RESULTS_DIR
    / "GSE96058_external_predictions_trained_on_GSE81538_405.csv"
)

SAVED_SUMMARY_PATH = (
    EXTERNAL_RESULTS_DIR
    / "GSE96058_external_validation_trained_on_GSE81538_405_summary.csv"
)


# Ensure the output folder exists.
REPRODUCIBILITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 2. Confirm that all required files exist
# =========================================================

required_files = [
    MODEL_BUNDLE_PATH,
    METADATA_PATH,
    SHARED_GENES_PATH,
    CHECKSUM_PATH,
    SAVED_PREDICTIONS_PATH,
    SAVED_SUMMARY_PATH,
]


for required_file in required_files:

    assert required_file.exists(), (
        f"Required file was not found:\n"
        f"{required_file}"
    )


print(
    "All required reproducibility files were found."
)


# =========================================================
# 3. Define the SHA-256 checksum function
# =========================================================

def calculate_sha256(file_path):
    """
    Calculate and return the SHA-256 checksum
    for one file.
    """

    hash_object = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as input_file:

        for block in iter(
            lambda: input_file.read(
                1024 * 1024
            ),
            b""
        ):

            hash_object.update(
                block
            )

    return hash_object.hexdigest()


# =========================================================
# 4. Verify the saved artifact checksums
# =========================================================

recorded_checksums = pd.read_csv(
    CHECKSUM_PATH
)


required_checksum_columns = {
    "filename",
    "sha256",
}

missing_checksum_columns = (
    required_checksum_columns
    .difference(
        recorded_checksums.columns
    )
)

assert not missing_checksum_columns, (
    "The checksum file is missing these columns: "
    f"{sorted(missing_checksum_columns)}"
)


artifact_paths = {
    MODEL_BUNDLE_PATH.name:
        MODEL_BUNDLE_PATH,

    METADATA_PATH.name:
        METADATA_PATH,

    SHARED_GENES_PATH.name:
        SHARED_GENES_PATH,
}


checksum_verification_records = []


for filename, artifact_path in artifact_paths.items():

    matching_rows = (
        recorded_checksums[
            recorded_checksums[
                "filename"
            ] == filename
        ]
    )

    assert len(matching_rows) == 1, (
        f"Expected one checksum record for "
        f"{filename}, but found "
        f"{len(matching_rows)}."
    )

    recorded_hash = (
        matching_rows[
            "sha256"
        ]
        .iloc[0]
    )

    current_hash = calculate_sha256(
        artifact_path
    )

    checksum_verification_records.append(
        {
            "filename":
                filename,

            "recorded_sha256":
                recorded_hash,

            "current_sha256":
                current_hash,

            "checksum_match":
                recorded_hash
                == current_hash,
        }
    )


checksum_verification = pd.DataFrame(
    checksum_verification_records
)


assert checksum_verification[
    "checksum_match"
].all(), (
    "At least one saved model artifact "
    "failed checksum verification."
)


print(
    "All saved model artifacts passed "
    "SHA-256 verification."
)


# =========================================================
# 5. Load the saved model bundle
# =========================================================

model_bundle = joblib.load(
    MODEL_BUNDLE_PATH
)


required_bundle_keys = {
    "pipeline",
    "shared_genes",
    "classification_threshold",
    "metadata",
}


missing_bundle_keys = (
    required_bundle_keys
    .difference(
        model_bundle.keys()
    )
)


assert not missing_bundle_keys, (
    "The model bundle is missing these keys: "
    f"{sorted(missing_bundle_keys)}"
)


reloaded_pipeline = (
    model_bundle[
        "pipeline"
    ]
)

bundle_shared_genes = list(
    model_bundle[
        "shared_genes"
    ]
)

classification_threshold = float(
    model_bundle[
        "classification_threshold"
    ]
)

bundle_metadata = (
    model_bundle[
        "metadata"
    ]
)


print(
    "Saved model bundle loaded successfully."
)


# =========================================================
# 6. Load and verify the supporting metadata files
# =========================================================

shared_gene_table = pd.read_csv(
    SHARED_GENES_PATH
)


with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as metadata_file:

    metadata_json = json.load(
        metadata_file
    )


assert "gene" in shared_gene_table.columns, (
    "The shared-gene file does not contain "
    "a column named 'gene'."
)

shared_genes_from_csv = (
    shared_gene_table[
        "gene"
    ]
    .astype(str)
    .tolist()
)


assert len(bundle_shared_genes) == 18213
assert len(shared_genes_from_csv) == 18213

assert bundle_shared_genes == (
    shared_genes_from_csv
)

assert np.isclose(
    classification_threshold,
    0.50
)

assert metadata_json[
    "development_sample_size"
] == 405

assert metadata_json[
    "development_er_negative"
] == 82

assert metadata_json[
    "development_er_positive"
] == 323

assert metadata_json[
    "external_validation_sample_size"
] == 3073

assert metadata_json[
    "number_of_shared_genes"
] == 18213

assert metadata_json[
    "number_of_selected_genes"
] == 1000

assert bundle_metadata == metadata_json


print(
    "Shared-gene order and model metadata "
    "were verified successfully."
)


# =========================================================
# 7. Load the unchanged GSE96058 external cohort
# =========================================================

X_external = pd.read_pickle(
    PROCESSED_DIR
    / "GSE96058_X_external_ER.pkl"
)

y_external = pd.read_pickle(
    PROCESSED_DIR
    / "GSE96058_y_external_ER.pkl"
)


assert X_external.shape == (
    3073,
    30865
)

assert len(y_external) == 3073

assert int(
    (y_external == 0).sum()
) == 241

assert int(
    (y_external == 1).sum()
) == 2832


print(
    "Unchanged GSE96058 external cohort loaded."
)


# =========================================================
# 8. Align external genes using the stored order
# =========================================================

missing_external_genes = [
    gene
    for gene in bundle_shared_genes
    if gene not in X_external.columns
]


assert len(missing_external_genes) == 0, (
    "Some required genes are missing from "
    "the external dataset."
)


X_external_aligned = (
    X_external
    .loc[
        :,
        bundle_shared_genes
    ]
    .copy()
)


assert X_external_aligned.shape == (
    3073,
    18213
)

assert (
    X_external_aligned
    .columns
    .tolist()
    == bundle_shared_genes
)


print(
    "External cohort aligned using the exact "
    "stored 18,213-gene order."
)


# =========================================================
# 9. Generate predictions in the clean environment
# =========================================================

clean_probabilities = (
    reloaded_pipeline
    .predict_proba(
        X_external_aligned
    )[:, 1]
)


clean_predictions = (
    clean_probabilities
    >= classification_threshold
).astype(int)


assert len(clean_probabilities) == 3073
assert len(clean_predictions) == 3073

assert np.isfinite(
    clean_probabilities
).all()

assert (
    clean_probabilities >= 0
).all()

assert (
    clean_probabilities <= 1
).all()


print(
    "Predictions generated successfully in "
    "the clean environment."
)


# =========================================================
# 10. Recalculate external-validation metrics
# =========================================================

y_external_array = np.asarray(
    y_external
).astype(int)


clean_confusion_matrix = confusion_matrix(
    y_external_array,
    clean_predictions,
    labels=[
        0,
        1
    ]
)


tn, fp, fn, tp = (
    clean_confusion_matrix.ravel()
)


clean_metrics = {
    "AUROC":
        roc_auc_score(
            y_external_array,
            clean_probabilities
        ),

    "AUPRC":
        average_precision_score(
            y_external_array,
            clean_probabilities
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_external_array,
            clean_predictions
        ),

    "sensitivity":
        tp / (
            tp + fn
        ),

    "specificity":
        tn / (
            tn + fp
        ),

    "brier_score":
        brier_score_loss(
            y_external_array,
            clean_probabilities
        ),

    "true_negatives":
        int(tn),

    "false_positives":
        int(fp),

    "false_negatives":
        int(fn),

    "true_positives":
        int(tp),
}


clean_metrics_table = pd.DataFrame(
    [
        clean_metrics
    ]
)


# =========================================================
# 11. Load the previously saved predictions
# =========================================================

saved_predictions = pd.read_csv(
    SAVED_PREDICTIONS_PATH
)


required_prediction_columns = {
    "predicted_probability_ER_positive",
    "predicted_ER_status",
}


missing_prediction_columns = (
    required_prediction_columns
    .difference(
        saved_predictions.columns
    )
)


assert not missing_prediction_columns, (
    "The saved prediction file is missing "
    f"these columns: "
    f"{sorted(missing_prediction_columns)}"
)


assert len(saved_predictions) == 3073


saved_probability_array = (
    saved_predictions[
        "predicted_probability_ER_positive"
    ]
    .to_numpy(
        dtype=float
    )
)


saved_class_array = (
    saved_predictions[
        "predicted_ER_status"
    ]
    .to_numpy(
        dtype=int
    )
)


# =========================================================
# 12. Compare clean and original predictions
# =========================================================

probability_differences = np.abs(
    clean_probabilities
    - saved_probability_array
)


maximum_probability_difference = float(
    probability_differences.max()
)


mean_probability_difference = float(
    probability_differences.mean()
)


number_of_prediction_disagreements = int(
    (
        clean_predictions
        != saved_class_array
    ).sum()
)


assert np.allclose(
    clean_probabilities,
    saved_probability_array,
    rtol=1e-10,
    atol=1e-12
), (
    "The clean-environment probabilities "
    "do not reproduce the saved probabilities."
)


assert number_of_prediction_disagreements == 0, (
    "The clean-environment classifications "
    "do not reproduce the saved classifications."
)


print(
    "Clean-environment predictions match "
    "the previously saved predictions."
)


# =========================================================
# 13. Compare metrics with the saved summary
# =========================================================

saved_summary = pd.read_csv(
    SAVED_SUMMARY_PATH
)


assert len(saved_summary) == 1


metric_columns = [
    "AUROC",
    "AUPRC",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "brier_score",
]


metric_comparison_records = []


for metric in metric_columns:

    saved_metric_value = float(
        saved_summary[
            metric
        ]
        .iloc[0]
    )

    clean_metric_value = float(
        clean_metrics[
            metric
        ]
    )

    metric_comparison_records.append(
        {
            "metric":
                metric,

            "saved_value":
                saved_metric_value,

            "clean_environment_value":
                clean_metric_value,

            "absolute_difference":
                abs(
                    saved_metric_value
                    - clean_metric_value
                ),

            "values_match":
                np.isclose(
                    saved_metric_value,
                    clean_metric_value,
                    rtol=1e-10,
                    atol=1e-12
                ),
        }
    )


metric_comparison = pd.DataFrame(
    metric_comparison_records
)


assert metric_comparison[
    "values_match"
].all(), (
    "At least one clean-environment metric "
    "does not match the saved metric."
)


# =========================================================
# 14. Verify expected reported values
# =========================================================

assert np.isclose(
    clean_metrics[
        "AUROC"
    ],
    0.9672,
    atol=0.0001
)

assert np.isclose(
    clean_metrics[
        "AUPRC"
    ],
    0.9966,
    atol=0.0001
)

assert np.isclose(
    clean_metrics[
        "balanced_accuracy"
    ],
    0.9341,
    atol=0.0001
)

assert np.isclose(
    clean_metrics[
        "sensitivity"
    ],
    0.9428,
    atol=0.0001
)

assert np.isclose(
    clean_metrics[
        "specificity"
    ],
    0.9253,
    atol=0.0001
)

assert np.isclose(
    clean_metrics[
        "brier_score"
    ],
    0.0510,
    atol=0.0001
)


assert clean_confusion_matrix.tolist() == [
    [
        223,
        18
    ],
    [
        162,
        2670
    ],
]


print(
    "External metrics and confusion matrix "
    "were reproduced successfully."
)


# =========================================================
# 15. Create the clean-environment verification report
# =========================================================

verification_summary = {
    "model_artifact_checksums_verified":
        True,

    "model_loaded_successfully":
        True,

    "shared_gene_order_verified":
        True,

    "external_sample_size":
        3073,

    "number_of_aligned_genes":
        18213,

    "classification_threshold":
        classification_threshold,

    "maximum_probability_difference":
        maximum_probability_difference,

    "mean_probability_difference":
        mean_probability_difference,

    "prediction_disagreements":
        number_of_prediction_disagreements,

    "AUROC":
        float(
            clean_metrics[
                "AUROC"
            ]
        ),

    "AUPRC":
        float(
            clean_metrics[
                "AUPRC"
            ]
        ),

    "balanced_accuracy":
        float(
            clean_metrics[
                "balanced_accuracy"
            ]
        ),

    "sensitivity":
        float(
            clean_metrics[
                "sensitivity"
            ]
        ),

    "specificity":
        float(
            clean_metrics[
                "specificity"
            ]
        ),

    "brier_score":
        float(
            clean_metrics[
                "brier_score"
            ]
        ),

    "true_negatives":
        int(tn),

    "false_positives":
        int(fp),

    "false_negatives":
        int(fn),

    "true_positives":
        int(tp),

    "reproducibility_status":
        "PASSED",
}


VERIFICATION_REPORT_PATH = (
    REPRODUCIBILITY_DIR
    / "clean_environment_model_verification.json"
)


with open(
    VERIFICATION_REPORT_PATH,
    "w",
    encoding="utf-8"
) as verification_file:

    json.dump(
        verification_summary,
        verification_file,
        indent=4
    )


CLEAN_METRICS_PATH = (
    REPRODUCIBILITY_DIR
    / "clean_environment_external_metrics.csv"
)


clean_metrics_table.to_csv(
    CLEAN_METRICS_PATH,
    index=False
)


CHECKSUM_VERIFICATION_PATH = (
    REPRODUCIBILITY_DIR
    / "clean_environment_checksum_verification.csv"
)


checksum_verification.to_csv(
    CHECKSUM_VERIFICATION_PATH,
    index=False
)


METRIC_COMPARISON_PATH = (
    REPRODUCIBILITY_DIR
    / "clean_environment_metric_comparison.csv"
)


metric_comparison.to_csv(
    METRIC_COMPARISON_PATH,
    index=False
)


# =========================================================
# 16. Display the verification results
# =========================================================

print(
    "\nClean-environment external metrics:"
)

display(
    clean_metrics_table.round(
        6
    )
)


print(
    "\nMetric comparison:"
)

display(
    metric_comparison.round(
        12
    )
)


print(
    "\nChecksum verification:"
)

display(
    checksum_verification
)


print(
    "\nClean-environment confusion matrix:"
)

print(
    clean_confusion_matrix
)


print(
    "\nMaximum probability difference:"
)

print(
    maximum_probability_difference
)


print(
    "\nMean probability difference:"
)

print(
    mean_probability_difference
)


print(
    "\nPrediction disagreements:"
)

print(
    number_of_prediction_disagreements
)


print(
    "\nClean-environment model reproducibility "
    "verification completed successfully."
)


print(
    "\nVerification report:"
)

print(
    VERIFICATION_REPORT_PATH
)


print(
    "\nReproducibility status: PASSED"
)

All required reproducibility files were found.
All saved model artifacts passed SHA-256 verification.
Saved model bundle loaded successfully.
Shared-gene order and model metadata were verified successfully.
Unchanged GSE96058 external cohort loaded.
External cohort aligned using the exact stored 18,213-gene order.
Predictions generated successfully in the clean environment.
Clean-environment predictions match the previously saved predictions.
External metrics and confusion matrix were reproduced successfully.

Clean-environment external metrics:


,AUROC,AUPRC,balanced_accuracy,sensitivity,specificity,brier_score,true_negatives,false_positives,false_negatives,true_positives
0,0.967171,0.99657,0.934054,0.942797,0.925311,0.050984,223,18,162,2670



Metric comparison:


,metric,saved_value,clean_environment_value,absolute_difference,values_match
0,AUROC,0.967171,0.967171,0.0,True
1,AUPRC,0.996570,0.996570,0.0,True
2,balanced_accuracy,0.934054,0.934054,0.0,True
3,sensitivity,0.942797,0.942797,0.0,True
4,specificity,0.925311,0.925311,0.0,True
5,brier_score,0.050984,0.050984,0.0,True



Checksum verification:


,filename,recorded_sha256,current_sha256,checksum_match
0,GSE81538_405_ER_prediction_model_bundle.joblib,1667a4e009f9c64b9e081b605e6bc7b4966ff7bc1df259...,1667a4e009f9c64b9e081b605e6bc7b4966ff7bc1df259...,True
1,GSE81538_405_ER_prediction_model_metadata.json,f0124b052fe5e9974f4f0de0e65fa25030464a92e79ea2...,f0124b052fe5e9974f4f0de0e65fa25030464a92e79ea2...,True
2,GSE81538_GSE96058_shared_genes_405.csv,fea7153d49e49456d0673b44c4cd0d16206ab147fc3691...,fea7153d49e49456d0673b44c4cd0d16206ab147fc3691...,True



Clean-environment confusion matrix:
[[ 223   18]
 [ 162 2670]]

Maximum probability difference:
1.1102230246251565e-16

Mean probability difference:
4.373965095692702e-17

Prediction disagreements:
0

Clean-environment model reproducibility verification completed successfully.

Verification report:
C:\GEO_Breast_Reliability\reproducibility\clean_environment_model_verification.json

Reproducibility status: PASSED
